# M2: FM100 group comparisons

`PLANScores.md` M2: `comparisons.compare_fm100_feature` (Mann-Whitney U + effect size, subject-pooled scores, same convention `beh/scripts/features.py`'s `compare_shape_feature` uses) across `comparisons.FEATURES` (TES, PES_RG, PES_BY, VKS_MajRad, VKS_MinRad, VKS_Angle) for the four requested pairs: CTR vs PD, HC vs protan, HC vs deutan, protan vs deutan.

`VKS_Angle` is periodic (folds to `[0, 180)`) -- compared with the same folded-Mann-Whitney approximation `beh/`'s `orientation_deg` already uses (valid as long as a group's angles don't straddle the 0/180 wrap point; see `comparisons.py`'s module docstring).

In [1]:
import sys
sys.path.append('../scripts')

import pandas as pd

import comparisons
import loader

df = loader.load_fm100_raw()

pairs = [
    ('CTR', None, 'PD', None),
    ('CTR', None, 'CVD', 'protan'),
    ('CTR', None, 'CVD', 'deutan'),
    ('CVD', 'protan', 'CVD', 'deutan'),
]
pair_labels = ['CTR vs PD', 'HC vs protan', 'HC vs deutan', 'protan vs deutan']

In [2]:
rows = []
for (g1, sg1, g2, sg2), label in zip(pairs, pair_labels):
    for feature in comparisons.FEATURES:
        result = comparisons.compare_fm100_feature(df, feature, group1=g1, subgroup1=sg1, group2=g2, subgroup2=sg2)
        rows.append({'comparison': label, **result})
results = pd.DataFrame(rows)
results

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
0,CTR vs PD,TES,29.5,0.029346,-0.590278,0.204861,24,6
1,CTR vs PD,PES_RG,24.5,0.014772,-0.659722,0.170139,24,6
2,CTR vs PD,PES_BY,29.0,0.027541,-0.597222,0.201389,24,6
3,CTR vs PD,VKS_MajRad,31.0,0.033120,-0.569444,0.215278,24,6
4,CTR vs PD,VKS_MinRad,29.0,0.024639,-0.597222,0.201389,24,6
5,CTR vs PD,VKS_Angle,73.0,0.979853,0.013889,0.506944,24,6
6,HC vs protan,TES,6.0,0.000041,-0.944444,0.027778,24,9
7,HC vs protan,PES_RG,0.0,0.000014,-1.000000,0.000000,24,9
8,HC vs protan,PES_BY,48.5,0.017064,-0.550926,0.224537,24,9
9,HC vs protan,VKS_MajRad,7.0,0.000049,-0.935185,0.032407,24,9


## Just the significant rows (p < 0.05, uncorrected -- matching every other group-comparison battery in this project, e.g. `beh/`'s M1)

In [3]:
results[results['p_value'] < 0.05].sort_values('p_value')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
16,HC vs deutan,VKS_MinRad,6.0,0.000006,-0.937500,0.031250,24,8
15,HC vs deutan,VKS_MajRad,8.0,0.000013,-0.916667,0.041667,24,8
7,HC vs protan,PES_RG,0.0,0.000014,-1.000000,0.000000,24,9
13,HC vs deutan,PES_RG,0.0,0.000032,-1.000000,0.000000,24,8
6,HC vs protan,TES,6.0,0.000041,-0.944444,0.027778,24,9
9,HC vs protan,VKS_MajRad,7.0,0.000049,-0.935185,0.032407,24,9
12,HC vs deutan,TES,7.5,0.000127,-0.921875,0.039062,24,8
10,HC vs protan,VKS_MinRad,17.0,0.000254,-0.842593,0.078704,24,9
14,HC vs deutan,PES_BY,24.0,0.001859,-0.750000,0.125000,24,8
11,HC vs protan,VKS_Angle,178.0,0.004964,0.648148,0.824074,24,9


## Holm-corrected, per pair (M3)

`comparisons.correct_multiple_comparisons` (Holm, the family-wise-error
default), applied once per comparison (6 features = one family), matching
how `ssvepBeh/`'s own correlation battery is corrected. Family-wise error
control is more conservative than the uncorrected view above -- some rows
significant there don't survive here.

In [4]:
corrected = pd.concat([comparisons.correct_multiple_comparisons(results[results['comparison'] == label]) for label in pair_labels])
corrected[corrected['significant']].sort_values('p_corrected')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2,p_corrected,significant
16,HC vs deutan,VKS_MinRad,6.0,0.000006,-0.937500,0.031250,24,8,0.000034,True
15,HC vs deutan,VKS_MajRad,8.0,0.000013,-0.916667,0.041667,24,8,0.000064,True
7,HC vs protan,PES_RG,0.0,0.000014,-1.000000,0.000000,24,9,0.000083,True
13,HC vs deutan,PES_RG,0.0,0.000032,-1.000000,0.000000,24,8,0.000129,True
9,HC vs protan,VKS_MajRad,7.0,0.000049,-0.935185,0.032407,24,9,0.000203,True
6,HC vs protan,TES,6.0,0.000041,-0.944444,0.027778,24,9,0.000203,True
12,HC vs deutan,TES,7.5,0.000127,-0.921875,0.039062,24,8,0.000382,True
10,HC vs protan,VKS_MinRad,17.0,0.000254,-0.842593,0.078704,24,9,0.000762,True
14,HC vs deutan,PES_BY,24.0,0.001859,-0.750000,0.125000,24,8,0.003717,True
11,HC vs protan,VKS_Angle,178.0,0.004964,0.648148,0.824074,24,9,0.009928,True
